# ⚡ Smart Power — Transformation

**Question:** When is electricity in the Netherlands cheapest **and** cleanest, and how much is driven by renewable generation?

The raw data (90 days, hourly, all in UTC) is pulled by `src/ingest.py`. This notebook
**loads that raw data**, transforms it, computes the renewable share, joins everything into
one clean table, and stores it in a MySQL database.

## 1. Load raw data (produced by `src/ingest.py`)

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from database import ensure_hourly_data_schema, get_mysql_engine

# Load the raw CSVs
weather = pd.read_csv(PROJECT_ROOT / "data" / "raw_weather.csv")
prices = pd.read_csv(PROJECT_ROOT / "data" / "raw_prices.csv")
gen = pd.read_csv(PROJECT_ROOT / "data" / "raw_generation.csv")
carbon = pd.read_csv(PROJECT_ROOT / "data" / "raw_carbon_intensity.csv")

# Make sure every timestamp is a proper UTC datetime
for name, source_df in [("weather", weather), ("prices", prices), ("generation", gen), ("carbon", carbon)]:
    source_df["timestamp"] = pd.to_datetime(source_df["timestamp"], utc=True)
    print(name, source_df.shape)

weather (2184, 3)
prices (2184, 2)
generation (8640, 11)
carbon (6915, 2)


## 2. Transform generation → hourly + renewable share

Generation is 15-min, so we resample it to hourly, then compute the **renewable share** for every hour — the key variable of the project.

In [2]:
# Resample generation from 15-min to hourly (average power per hour)
gen = gen.set_index("timestamp")
gen_hourly = gen.resample("h").mean()

print(gen_hourly.shape)   # ~2160 rows = 90 days x 24 hours
gen_hourly.head()

(2160, 10)


,Biomass,Fossil Gas,Fossil Hard coal,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Waste,Wind Offshore,Wind Onshore
timestamp,,,,,,,,,,
2026-05-05 22:00:00+00:00,19.96550,6302.35675,2278.71450,0.0,0.0,1997.01675,0.0,261.75600,1974.55050,818.53700
2026-05-05 23:00:00+00:00,22.50475,4903.80900,2271.07600,0.0,0.0,1898.88175,0.0,263.13375,1866.97375,859.98575
2026-05-06 00:00:00+00:00,26.25700,4376.67800,2237.65100,0.0,0.0,1887.43550,0.0,262.67700,1945.22975,860.59675
2026-05-06 01:00:00+00:00,26.30975,4146.75825,2273.05400,0.0,0.0,1826.83950,0.0,262.98300,2088.13700,862.65500
2026-05-06 02:00:00+00:00,26.35425,4106.26875,2279.21425,0.0,0.0,1878.63500,0.0,266.13725,2199.99125,865.47475


In [3]:
# Compute the renewable share for every hour
production_cols = gen_hourly.columns.tolist()          # all generation types

# Pick renewable columns by name (Wind Offshore, Wind Onshore, Solar, Hydro..., Biomass)
renewable_keywords = ["Solar", "Wind", "Hydro", "Biomass"]
renewable_cols = [c for c in production_cols
                  if any(k in c for k in renewable_keywords)]
print("Renewable columns:", renewable_cols)

gen_hourly["total_generation"]     = gen_hourly[production_cols].sum(axis=1)
gen_hourly["renewable_generation"] = gen_hourly[renewable_cols].sum(axis=1)
gen_hourly["renewable_share"]      = (
    gen_hourly["renewable_generation"] / gen_hourly["total_generation"]
)

gen_hourly[["total_generation", "renewable_generation", "renewable_share"]].head()

Renewable columns: ['Biomass', 'Hydro Run-of-river and poundage', 'Solar', 'Wind Offshore', 'Wind Onshore']


,total_generation,renewable_generation,renewable_share
timestamp,,,
2026-05-05 22:00:00+00:00,13652.89700,2813.05300,0.206041
2026-05-05 23:00:00+00:00,12086.36475,2749.46425,0.227485
2026-05-06 00:00:00+00:00,11596.52500,2832.08350,0.244218
2026-05-06 01:00:00+00:00,11486.73650,2977.10175,0.259177
2026-05-06 02:00:00+00:00,11622.07550,3091.82025,0.266030


In [4]:
# Sanity check: renewable_share should sit between 0 and 1
print(gen_hourly["renewable_share"].describe())

count    2160.000000
mean        0.198737
std         0.146982
min         0.000164
25%         0.084932
50%         0.162338
75%         0.279118
max         0.690521
Name: renewable_share, dtype: float64


## 3. Join + store in a database

Merge the three hourly tables on `timestamp` into one clean dataset, then save it to MySQL and check it with SQL (Unit 4 revision).

In [5]:
# Bring gen_hourly's index back as a column so we can merge on it
gen_hourly = gen_hourly.reset_index()

# Resample Wattnet's validated 15-minute values to an hourly mean.
carbon_hourly = (carbon.set_index("timestamp")["carbon_intensity_gco2_kwh"]
                 .resample("h").agg(["mean", "count"]))
carbon_hourly = (carbon_hourly.loc[carbon_hourly["count"] == 4, ["mean"]]
                 .rename(columns={"mean": "carbon_intensity_gco2_kwh"})
                 .reset_index())
assert carbon_hourly["carbon_intensity_gco2_kwh"].ge(0).all()

# Merge the four tables on timestamp.
# how="inner" keeps timestamps present in all four tables.
# Source rows can still contain null measurements; those are handled below.
df = prices.merge(gen_hourly, on="timestamp", how="inner")
df = df.merge(weather, on="timestamp", how="inner")
df = df.merge(carbon_hourly, on="timestamp", how="inner")

print(df.shape)   # ~2150 rows (the overlapping hours)
df.head()

(1698, 18)


,timestamp,electricity_price,Biomass,Fossil Gas,Fossil Hard coal,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Waste,Wind Offshore,Wind Onshore,total_generation,renewable_generation,renewable_share,wind_speed,solar_radiation,carbon_intensity_gco2_kwh
0,2026-05-06 00:00:00+00:00,0.13,26.25700,4376.67800,2237.65100,0.0,0.0,1887.43550,0.00000,262.67700,1945.22975,860.59675,11596.5250,2832.08350,0.244218,NaN,NaN,445.5400
1,2026-05-06 01:00:00+00:00,0.13,26.30975,4146.75825,2273.05400,0.0,0.0,1826.83950,0.00000,262.98300,2088.13700,862.65500,11486.7365,2977.10175,0.259177,NaN,NaN,440.6775
2,2026-05-06 02:00:00+00:00,0.13,26.35425,4106.26875,2279.21425,0.0,0.0,1878.63500,0.00000,266.13725,2199.99125,865.47475,11622.0755,3091.82025,0.266030,NaN,NaN,435.6500
3,2026-05-06 03:00:00+00:00,0.15,46.66050,4337.70700,2274.97050,0.0,0.0,2243.83075,0.00000,257.54950,2260.01175,881.41950,12302.1495,3188.09175,0.259149,NaN,NaN,431.6425
4,2026-05-06 04:00:00+00:00,0.16,199.04875,4523.28925,2159.12450,0.0,0.0,2781.07850,4.27325,245.96225,2389.63275,854.74475,13157.1540,3447.69950,0.262040,NaN,NaN,412.4625


In [6]:
# Keep the core columns for analysis + machine learning
clean = df[[
    "timestamp",
    "electricity_price",
    "renewable_share",
    "renewable_generation",
    "total_generation",
    "carbon_intensity_gco2_kwh",
    "wind_speed",
    "solar_radiation",
]].copy()

print(clean.shape)
clean.head()

(1698, 8)


,timestamp,electricity_price,renewable_share,renewable_generation,total_generation,carbon_intensity_gco2_kwh,wind_speed,solar_radiation
0,2026-05-06 00:00:00+00:00,0.13,0.244218,2832.08350,11596.5250,445.5400,NaN,NaN
1,2026-05-06 01:00:00+00:00,0.13,0.259177,2977.10175,11486.7365,440.6775,NaN,NaN
2,2026-05-06 02:00:00+00:00,0.13,0.266030,3091.82025,11622.0755,435.6500,NaN,NaN
3,2026-05-06 03:00:00+00:00,0.15,0.259149,3188.09175,12302.1495,431.6425,NaN,NaN
4,2026-05-06 04:00:00+00:00,0.16,0.262040,3447.69950,13157.1540,412.4625,NaN,NaN


In [7]:
from sqlalchemy import text

# Keep only complete hours shared by all four sources.
rows_before = len(clean)
clean = clean.dropna().reset_index(drop=True)
print(f"Removed {rows_before - len(clean):,} incomplete rows; {len(clean):,} complete rows remain.")

assert not clean["timestamp"].duplicated().any(), "Duplicate timestamps found"
assert clean.isna().sum().sum() == 0, "Missing values remain"
assert clean["renewable_share"].between(0, 1).all(), "Renewable share outside [0, 1]"
assert clean["carbon_intensity_gco2_kwh"].ge(0).all(), "Negative carbon intensity"

# MySQL DATETIME is timezone-naive; the values remain explicitly defined as UTC.
clean["timestamp"] = clean["timestamp"].dt.tz_localize(None)
clean = clean.rename(columns={"timestamp": "timestamp_utc"})

engine = get_mysql_engine(PROJECT_ROOT)
ensure_hourly_data_schema(engine)
with engine.begin() as connection:
    connection.execute(text("DELETE FROM hourly_data"))
    clean.to_sql(
        "hourly_data", connection, if_exists="append", index=False,
        chunksize=500, method="multi",
    )

print("Saved clean table to MySQL database: smart_power.hourly_data")

Removed 528 incomplete rows; 1,170 complete rows remain.


Saved clean table to MySQL database: smart_power.hourly_data


In [8]:
# Read it back with SQL to confirm
display(pd.read_sql("SELECT * FROM hourly_data LIMIT 5", engine))

display(pd.read_sql("""
    SELECT COUNT(*)                          AS total_rows,
           ROUND(AVG(electricity_price), 3)  AS avg_price,
           ROUND(AVG(renewable_share), 3)    AS avg_renewable
    FROM hourly_data
""", engine))

,timestamp_utc,electricity_price,renewable_share,renewable_generation,total_generation,carbon_intensity_gco2_kwh,wind_speed,solar_radiation
0,2026-05-28 00:00:00,0.15,0.224326,2434.29775,10851.62950,362.6350,12.5,0.0
1,2026-05-28 01:00:00,0.15,0.214828,2264.14525,10539.35550,364.6325,11.2,0.0
2,2026-05-28 02:00:00,0.16,0.205682,2135.27825,10381.46350,366.8150,10.6,0.0
3,2026-05-28 03:00:00,0.16,0.197786,2015.51175,10190.36625,369.7850,9.5,0.0
4,2026-05-28 04:00:00,0.18,0.169004,1846.50025,10925.76175,392.6750,9.3,5.0


,total_rows,avg_price,avg_renewable
0,1170,0.13,0.215


In [9]:
# Final in-memory data-quality summary
display(clean.isna().sum().to_frame("missing_values"))
display(clean.describe().T)

,missing_values
timestamp_utc,0
electricity_price,0
renewable_share,0
renewable_generation,0
total_generation,0
carbon_intensity_gco2_kwh,0
wind_speed,0
solar_radiation,0


,count,mean,min,25%,50%,75%,max,std
timestamp_utc,1170,2026-07-04 09:21:35.384615,2026-05-28 00:00:00,2026-06-09 11:15:00,2026-07-09 07:30:00,2026-07-21 16:45:00,2026-08-03 21:00:00,NaN
electricity_price,1170.0,0.130231,-0.05,0.08,0.15,0.18,0.67,0.078251
renewable_share,1170.0,0.215244,0.000164,0.090587,0.182821,0.301661,0.690521,0.156937
renewable_generation,1170.0,1840.043563,1.721,756.797812,1550.61325,2537.70325,6500.93725,1395.738488
total_generation,1170.0,9699.975578,1802.1115,6613.423,10246.03175,12603.240875,20916.41125,4181.023077
carbon_intensity_gco2_kwh,1170.0,402.906521,86.82,341.821875,421.7875,492.63875,682.4025,117.441669
wind_speed,1170.0,11.827009,0.4,8.6,11.5,14.8,28.1,4.673272
solar_radiation,1170.0,242.905128,0.0,0.0,108.0,458.25,880.0,280.541


In [10]:
# Cross-check that MySQL contains exactly the validated in-memory dataset.
database_count = pd.read_sql("SELECT COUNT(*) AS total_rows FROM hourly_data", engine).loc[0, "total_rows"]
assert database_count == len(clean)
print(f"Database validation passed: {database_count:,} rows.")

Database validation passed: 1,170 rows.


In [11]:
preview = pd.read_sql(
    "SELECT * FROM hourly_data ORDER BY timestamp_utc LIMIT 10", engine
)
engine.dispose()
preview

,timestamp_utc,electricity_price,renewable_share,renewable_generation,total_generation,carbon_intensity_gco2_kwh,wind_speed,solar_radiation
0,2026-05-28 00:00:00,0.15,0.224326,2434.29775,10851.62950,362.6350,12.5,0.0
1,2026-05-28 01:00:00,0.15,0.214828,2264.14525,10539.35550,364.6325,11.2,0.0
2,2026-05-28 02:00:00,0.16,0.205682,2135.27825,10381.46350,366.8150,10.6,0.0
3,2026-05-28 03:00:00,0.16,0.197786,2015.51175,10190.36625,369.7850,9.5,0.0
4,2026-05-28 04:00:00,0.18,0.169004,1846.50025,10925.76175,392.6750,9.3,5.0
5,2026-05-28 05:00:00,0.17,0.118318,1467.24575,12400.88625,428.8325,8.9,76.0
6,2026-05-28 06:00:00,0.15,0.096214,1297.05125,13480.96300,470.0900,7.4,209.0
7,2026-05-28 07:00:00,0.11,0.092129,1269.26525,13777.03400,472.5375,7.4,361.0
8,2026-05-28 08:00:00,0.02,0.085706,1328.15200,15496.61650,485.7925,8.5,516.0
9,2026-05-28 09:00:00,0.00,0.065827,1111.41300,16883.77450,498.2950,8.5,655.0
